## Investment Strategy Suggestions

- Buy stocks with consistently positive sentiment and strong momentum.
- Avoid stocks receiving highly negative sentiment.
- Combine sentiment analysis with RSI and MACD for stronger signals.
- Use lagged sentiment to anticipate next-day movements.

In [ ]:
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.stats import pearsonr


# =========================
# STEP 1 — LOAD DATA
# =========================
def load_data():
    news = pd.read_csv("../data/raw/news.csv")
    stocks = pd.read_csv("../data/raw/stock_prices.csv")
    return news, stocks


# =========================
# STEP 2 — SENTIMENT ANALYSIS
# =========================
def add_sentiment(news):

    analyzer = SentimentIntensityAnalyzer()

    def get_sentiment(text):
        return analyzer.polarity_scores(str(text))['compound']

    news['sentiment_score'] = news['headline'].apply(get_sentiment)

    return news


# =========================
# STEP 3 — DATE PROCESSING
# =========================
def process_dates(news, stocks):

    news['date'] = pd.to_datetime(news['date'], errors='coerce')
    stocks['Date'] = pd.to_datetime(stocks['Date'], errors='coerce')

    news['day'] = news['date'].dt.date
    stocks['day'] = stocks['Date'].dt.date

    return news, stocks


# =========================
# STEP 4 — HEADLINE LENGTH (EDA FEATURE)
# =========================
def add_headline_length(news):

    news['headline_length'] = news['headline'].astype(str).apply(len)

    return news


# =========================
# STEP 5 — DAILY SENTIMENT
# =========================
def daily_sentiment(news):

    return news.groupby(
        ['stock', 'day']
    )['sentiment_score'].mean().reset_index()


# =========================
# STEP 6 — STOCK RETURNS
# =========================
def compute_returns(stocks):

    stocks = stocks.sort_values('Date')

    stocks['Daily_Return'] = stocks['Adj Close'].pct_change()

    return stocks


# =========================
# STEP 7 — MERGE DATASETS
# =========================
def merge_data(sentiment_df, stocks):

    merged = pd.merge(
        sentiment_df,
        stocks,
        on='day',
        how='inner'
    )

    return merged


# =========================
# STEP 8 — OVERALL CORRELATION
# =========================
def overall_correlation(merged):

    merged = merged.dropna()

    corr, p_value = pearsonr(
        merged['sentiment_score'],
        merged['Daily_Return']
    )

    return corr, p_value


# =========================
# STEP 9 — STOCK LEVEL CORRELATION
# =========================
def stock_correlation(merged):

    return merged.groupby('stock').apply(
        lambda x: x['sentiment_score'].corr(x['Daily_Return'])
    )


# =========================
# STEP 10 — TIME ANALYSIS
# =========================
def time_series(merged):

    daily_sent = merged.groupby('day')['sentiment_score'].mean()
    daily_ret = merged.groupby('day')['Daily_Return'].mean()

    return daily_sent, daily_ret


# =========================
# STEP 11 — RUN FULL PIPELINE
# =========================
def run_pipeline():

    # Load data
    news, stocks = load_data()

    # Sentiment
    news = add_sentiment(news)

    # EDA feature (headline length)
    news = add_headline_length(news)

    # Dates
    news, stocks = process_dates(news, stocks)

    # Daily sentiment
    sentiment_df = daily_sentiment(news)

    # Returns
    stocks = compute_returns(stocks)

    # Merge
    merged = merge_data(sentiment_df, stocks)

    # Correlations
    corr, p_value = overall_correlation(merged)
    stock_corr = stock_correlation(merged)

    # Time series
    daily_sent, daily_ret = time_series(merged)

    return {
        "merged": merged,
        "overall_corr": corr,
        "p_value": p_value,
        "stock_corr": stock_corr,
        "daily_sentiment": daily_sent,
        "daily_returns": daily_ret
    }


# =========================
# STEP 12 — RUN SCRIPT
# =========================
if __name__ == "__main__":

    results = run_pipeline()

    print("\n=========================")
    print("OVERALL CORRELATION")
    print("=========================")

    print("Correlation:", results["overall_corr"])
    print("P-value:", results["p_value"])

    print("\n=========================")
    print("TOP STOCK CORRELATIONS")
    print("=========================")

    print(results["stock_corr"].sort_values(ascending=False).head(10))

    print("\n=========================")
    print("BOTTOM STOCK CORRELATIONS")
    print("=========================")

    print(results["stock_corr"].sort_values().head(10))